In [5]:
import pandas as pd
import pickle
import re
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk
from flask import Flask, render_template, request
from googleapiclient.discovery import build

In [3]:
from googleapiclient.discovery import build

In [1]:
!pip install vaderSentiment


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

# Load dataset
df = pd.read_csv(r"C:\Users\Nishitha reddy\Downloads\youtube_comments_cleaned.csv", encoding='utf-8')

# Only keep relevant columns
df = df[['CommentText', 'Sentiment']].dropna()

# Map Sentiment to numerical labels
sentiment_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
df['target'] = df['Sentiment'].map(sentiment_map)

# Optional sampling for speed
df = df.sample(10000, random_state=42).reset_index(drop=True)

# Initialize VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Preprocessing function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|@\w+|#\w+", '', text)
    return text.strip()

df['clean_text'] = df['CommentText'].apply(clean_text)

# Predict sentiment using VADER
vader_preds = []

for text in tqdm(df['clean_text'], desc="Analyzing with VADER"):
    score = analyzer.polarity_scores(text)['compound']
    if score >= 0.05:
        vader_preds.append(2)  # Positive
    elif score <= -0.05:
        vader_preds.append(0)  # Negative
    else:
        vader_preds.append(1)  # Neutral

# Evaluation
print("Accuracy:", accuracy_score(df['target'], vader_preds) * 100)
print("\nClassification Report:")
print(classification_report(df['target'], vader_preds, target_names=['Negative', 'Neutral', 'Positive']))


Analyzing with VADER: 100%|████████████████████████████████████████████████████| 10000/10000 [00:03<00:00, 3052.29it/s]


Accuracy: 53.190000000000005

Classification Report:
              precision    recall  f1-score   support

    Negative       0.63      0.43      0.51      3319
     Neutral       0.51      0.40      0.45      3316
    Positive       0.50      0.76      0.60      3365

    accuracy                           0.53     10000
   macro avg       0.55      0.53      0.52     10000
weighted avg       0.54      0.53      0.52     10000



In [6]:
import pandas as pd
import re
import nltk
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from nltk.corpus import stopwords

# Download stopwords
nltk.download('stopwords')

# Load Dataset
df = pd.read_csv(r"C:\Users\Nishitha reddy\Downloads\youtube_comments_cleaned.csv")  # Change path if needed

# Normalize and clean the Sentiment column
df['Sentiment'] = df['Sentiment'].astype(str).str.strip().str.lower()
df = df[df['Sentiment'].isin(['positive', 'neutral', 'negative'])]

# Preprocessing function
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r"[^a-zA-Z\s]", '', text)
    stop_words = set(stopwords.words('english'))
    words = [word for word in text.split() if word not in stop_words]
    return ' '.join(words)

# Clean comments
df['cleaned_comment'] = df['CommentText'].apply(preprocess_text)

# Encode sentiment
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['SentimentEncoded'] = df['Sentiment'].map(label_map)

# Feature and target
X = df['cleaned_comment']
y = df['SentimentEncoded']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF vectorizer
tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Logistic Regression model
lr_model = LogisticRegression(max_iter=200, multi_class='multinomial', solver='lbfgs')
lr_model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = lr_model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred) * 100)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

# ===  Pickle the model and vectorizer ===
with open("youtube_sentiment_model.pkl", "wb") as model_file:
    pickle.dump(lr_model, model_file)

with open("youtube_tfidf_vectorizer.pkl", "wb") as vec_file:
    pickle.dump(tfidf, vec_file)

print("\nModel and vectorizer saved successfully as .pkl files.")


[nltk_data] Downloading package stopwords to C:\Users\Nishitha
[nltk_data]     reddy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
C:\Users\Nishitha reddy\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 66.1842137130955

Classification Report:
               precision    recall  f1-score   support

    Negative       0.67      0.66      0.66     69194
     Neutral       0.59      0.66      0.62     68724
    Positive       0.75      0.67      0.71     68527

    accuracy                           0.66    206445
   macro avg       0.67      0.66      0.66    206445
weighted avg       0.67      0.66      0.66    206445


Model and vectorizer saved successfully as .pkl files.


In [7]:
import pandas as pd

# Load the dataset
df = pd.read_csv('youtube_comments_cleaned.csv')  # Replace with your actual file path if needed

# Drop missing values (optional but safe)
df = df.dropna(subset=['CommentText', 'Sentiment'])

# Standardize sentiment labels (e.g., remove spaces, lower case)
df['Sentiment'] = df['Sentiment'].astype(str).str.strip().str.lower()

# Filter only neutral comments
neutral_comments = df[df['Sentiment'] == 'neutral']

# Save to a new CSV
neutral_comments[['CommentText']].to_csv('neutral_comments.csv', index=False, encoding='utf-8-sig')

print("✅ Neutral comments saved to 'neutral_comments.csv'")


✅ Neutral comments saved to 'neutral_comments.csv'
